In [ ]:
import os
import pandas as pd
from scipy.stats import norm

In [ ]:
font_size = 18

r4_reproducibility_rate = .86
r4_irreproducibility_rate = .14

r3_reproducibility_rate = .33
r3_irreproducibility_rate = .67

label_counts = {
    "r4_code_data_shared": 0,
    "r3_only_data_shared": 0,
    "no_data_shared": 0,
    "total": 0,
}

In [ ]:
exclude_theoretical = True

# Open llm_results.csv
csv_file_path = "../../results/llm_results.csv"

if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"CSV file not found: {csv_file_path}")

df = pd.read_csv(csv_file_path)

In [ ]:
len(df)

In [ ]:
if exclude_theoretical:
    # remove all of the rows where research_type is 1
    df = df[df["research_type"] != 1]

In [ ]:
len(df)

In [ ]:
label_results = {}

label_results = {
    2014: label_counts.copy(),
    2015: label_counts.copy(),
    2016: label_counts.copy(),
    2017: label_counts.copy(),
    2018: label_counts.copy(),
    2019: label_counts.copy(),
    2020: label_counts.copy(),
    2021: label_counts.copy(),
    2022: label_counts.copy(),
    2023: label_counts.copy(),
    2024: label_counts.copy(),
}

In [ ]:
df.head()

In [ ]:
for _, row in df.iterrows():
    year = row["year"]

    # R4 Shared code and data
    if row["open_source_code"] == 1 and row["train"] == 1:
        label_results[year]["r4_code_data_shared"] += 1
    # R3 Shared data only
    elif row["open_source_code"] == 0 and row["train"] == 1:
        label_results[year]["r3_only_data_shared"] += 1
    # No data shared
    else:
        label_results[year]["no_data_shared"] += 1

    label_results[year]["total"] += 1

In [ ]:

ci_interval_results = {}

"""
Reproducibility rate 95% CI using Wilson score intervals.

Gundersen et al. [41] empirical rates:
  - code + data shared: 6/7 reproducible
  - data only shared:   5/15 reproducible

For each year, the estimated reproducibility rate is:
  R = (r4 / total) * p_code + (r3 / total) * p_data

The weights (r4/total, r3/total) are fixed known counts. Since R is a linear
combination of the two base rates, the Wilson score CIs propagate analytically:
  R_lo = (r4 / total) * p_code_lo + (r3 / total) * p_data_lo
  R_hi = (r4 / total) * p_code_hi + (r3 / total) * p_data_hi
"""
# Gundersen et al. [41] base rates
N_CODE = 7;  K_CODE = 6   # code + data: 6/7 reproducible
N_DATA = 15; K_DATA = 5   # data only:   5/15 reproducible

# ---------------------------------------------------------------------------
# Wilson score interval
# ---------------------------------------------------------------------------

def wilson_ci(k, n, alpha=0.05):
    """Return (point, lower, upper) Wilson score CI for k successes out of n trials."""
    z = norm.ppf(1 - alpha / 2)
    p_hat = k / n
    center = (p_hat + z**2 / (2 * n)) / (1 + z**2 / n)
    margin = (z / (1 + z**2 / n)) * (p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) ** 0.5
    return p_hat, max(0.0, center - margin), min(1.0, center + margin)

p_code, p_code_lo, p_code_hi = wilson_ci(K_CODE, N_CODE)
p_data, p_data_lo, p_data_hi = wilson_ci(K_DATA, N_DATA)

print("=" * 70)
print("Base rate Wilson 95% CIs (from Gundersen et al. [41])")
print(f"  Code + data: {p_code:.4f}  [{p_code_lo:.4f}, {p_code_hi:.4f}]  (6/7)")
print(f"  Data only:   {p_data:.4f}  [{p_data_lo:.4f}, {p_data_hi:.4f}]  (5/15)")
print()

# ---------------------------------------------------------------------------
# Per-year calculation
# ---------------------------------------------------------------------------

print(f"{'Year':>6}  {'R (%)':>7}  {'95% CI':>18}")
print("-" * 35)

for year in sorted(label_results.keys()):
    d = label_results[year]
    w_code = d["r4_code_data_shared"] / d["total"]
    w_data = d["r3_only_data_shared"] / d["total"]

    R_point = w_code * p_code + w_data * p_data
    R_lo    = w_code * p_code_lo + w_data * p_data_lo
    R_hi    = w_code * p_code_hi + w_data * p_data_hi

    print(f"{year:>6}  {R_point*100:>6.1f}%  [{R_lo*100:.1f}%, {R_hi*100:.1f}%]")

    ci_interval_results[year] = (R_lo, R_hi)

In [ ]:
# Calculate reproducibility rates
for year in label_results:
    label_results[year]["r4_code_data_shared_reproducible"] = round(
        (label_results[year]["r4_code_data_shared"] / label_results[year]["total"]) * r4_reproducibility_rate, 2
    )
    label_results[year]["r4_code_data_shared_irreproducible"] = round(
        (label_results[year]["r4_code_data_shared"] / label_results[year]["total"]) * r4_irreproducibility_rate, 2
    )

    label_results[year]["r3_only_data_shared_reproducible"] = round(
        (label_results[year]["r3_only_data_shared"] / label_results[year]["total"]) * r3_reproducibility_rate, 2    
    )
    label_results[year]["r3_only_data_shared_irreproducible"] = round(
        (label_results[year]["r3_only_data_shared"] / label_results[year]["total"]) * r3_irreproducibility_rate, 2
    )
    label_results[year]["no_data_shared_rate"] = round(
        (label_results[year]["no_data_shared"] / label_results[year]["total"]), 2
    )

    # remove "r4_code_data_shared" and "r3_only_data_shared" from the label_results dictionary
    del label_results[year]["r4_code_data_shared"]
    del label_results[year]["r3_only_data_shared"]
    del label_results[year]["no_data_shared"]
    del label_results[year]["total"]

In [ ]:
# Make sure the sum of the rates is 1 for each year
for year in label_results:
    # verify that the sum of the rates is 1
    total_rate = round((
        label_results[year]["r4_code_data_shared_reproducible"]
        + label_results[year]["r4_code_data_shared_irreproducible"]
        + label_results[year]["r3_only_data_shared_reproducible"]
        + label_results[year]["r3_only_data_shared_irreproducible"]
        + label_results[year]["no_data_shared_rate"]
    ), 2)
    print(f"Year: {year}, Total Rate: {total_rate}")

    if total_rate < 1.0:
        label_results[year]["no_data_shared_rate"] += round(1.0 - total_rate, 2)

In [ ]:
from matplotlib.ticker import FuncFormatter

# Convert label_results to a DataFrame
df = pd.DataFrame(label_results).T

# Reorder the columns in the desired order
df = df[[
    "r3_only_data_shared_reproducible",
    "r4_code_data_shared_reproducible",
    "r3_only_data_shared_irreproducible",
    "r4_code_data_shared_irreproducible",
    "no_data_shared_rate",
]]

# Plot the stacked area chart
ax = df.plot(kind="area", stacked=True, figsize=(14, 7), color={
    "r3_only_data_shared_reproducible": "#f3b23e",
    "r4_code_data_shared_reproducible": "#e37137",
    "r3_only_data_shared_irreproducible": "#d62728",
    "r4_code_data_shared_irreproducible": "#e662bd",
    "no_data_shared_rate": "#5dbaf7",
})

# Set x-axis limits to include 2014 and 2024 flush with the y-axis
ax.set_xlim(2014, 2024)
ax.set_ylim(0, 1)

# Convert y-axis to percentage
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y * 100)}%"))

# Set y-axis ticks every 10%
ax.set_yticks([i / 10 for i in range(0, 11)])

# Add a secondary y-axis on the right
ax_right = ax.twinx()
ax_right.set_ylim(0, 1)

# Convert the right y-axis to percentage as well
ax_right.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y * 100)}%"))

# Set y-axis ticks every 10% for the right axis
ax_right.set_yticks([i / 10 for i in range(0, 11)])

# Set the x-axis ticks to be every year from 2014 to 2024
ax.set_xticks(range(2014, 2025))

# Add a semi-transparent confidence band for the estimated reproducibility rate
ci_df = pd.DataFrame.from_dict(ci_interval_results, orient="index", columns=["ci_lower", "ci_upper"]).sort_index()
#ax.fill_between(ci_df.index, ci_df["ci_lower"], ci_df["ci_upper"], color="#222222", alpha=0.33, linewidth=0, label="95% CI for Reproducibility Rate", zorder=2)
ax.fill_between(ci_df.index, ci_df["ci_lower"], ci_df["ci_upper"], facecolor="none", hatch="///", edgecolor="black", linewidth=0, label="95% CI for Reproducibility Rate", zorder=2)



# Add a black line tracing the boundary between no_data_shared_rate and r4_code_data_shared_irreproducible
df["public_datasets"] = df["r3_only_data_shared_reproducible"] + df["r4_code_data_shared_reproducible"] + df["r3_only_data_shared_irreproducible"] + df["r4_code_data_shared_irreproducible"]
ax.plot(df.index, df["public_datasets"], color="black", linestyle='--', linewidth=2, label="Percent Papers with Public Datasets", zorder=4)

# Add a black line tracing the boundary between r4_code_data_shared_reproducible and r3_only_data_shared_irreproducible
df["reproducible"] = df["r3_only_data_shared_reproducible"] + df["r4_code_data_shared_reproducible"]
ax.plot(df.index, df["reproducible"], color="black", linestyle='-', linewidth=2, label="Percent Reproducible Papers", zorder=4)

# Set the title and labels
ax.set_title("Estimated Reproducibility Rate of Papers based on Open Code and Datasets Sharing")
ax.set_ylabel("Percentage of Empirical Papers")
ax.set_xlabel("Year")

# Increase font size for better readability
ax.title.set_fontsize(font_size)
ax.xaxis.label.set_fontsize(font_size - 2)
ax.yaxis.label.set_fontsize(font_size - 2)
ax.tick_params(axis='x', labelsize=font_size - 4)
ax.tick_params(axis='y', labelsize=font_size - 4)
ax_right.tick_params(axis='y', labelsize=font_size - 4)

ax.set_prop_cycle(None)  # Reset color cycle for legend mapping

# Update legend labels for better readability
legend_labels = {
    "r3_only_data_shared_reproducible": "Estimated Reproducible - Only Open Datasets Shared",
    "r4_code_data_shared_reproducible": "Estimated Reproducible - Open Code & Datasets Shared",
    "r3_only_data_shared_irreproducible": "Estimated Irreproducible - Only Open Datasets Shared",
    "r4_code_data_shared_irreproducible": "Estimated Irreproducible - Open Code & Datasets Shared",
    "no_data_shared_rate": "No Data Shared - Unknown Reproducibility Rate",
    "95% CI for Reproducibility Rate": "95% Confidence Interval for Reproducibility Rate",
    "Percent Papers with Public Datasets": "Percentage of Papers with Open Datasets",
    "Percent Reproducible Papers": "Percentage of Reproducible Papers"
}

handles, labels = ax.get_legend_handles_labels()
new_labels = [legend_labels.get(label, label) for label in labels]
ax.legend(handles, new_labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=font_size - 4)


ax.figure.savefig("../../plots/reproducibility_rate.pdf", format="pdf", bbox_inches="tight")